# To be run in env `isce2`

In [ ]:
import h5py
import matplotlib.pyplot as plt
from pathlib import Path
import os

from numpy import NaN

gunwfile='/Volumes/T9_InSAR/2026-04-14_nevada/NISAR/postseismic/NISAR_L2_PR_GUNW_018_106_A_021_019_4000_SH_20260422T130117_20260422T130152_20260504T130117_20260504T130152_P05023_N_F_J_001.h5'
directory = Path(gunwfile).parent

# NISAR_<Level>_<Proc>_<ProductType>_<Cycle>_<Track>_<Dir>_<Frame>_<Mode>_<OptionalParameters>_<Version>.h5
# NISAR_L2_PR_GUNW_022_162_A_007_023_4000_SH_20260613T100656_20260613T100731_20260625T100655_20260625T100730_P05023_N_F_J_001.h5

# Parse the filename to extract metadata
filename = Path(gunwfile).name.split('.')[0]    # Remove the .h5
parts = filename.split('_')

# print(parts)
metadata = {
    'Sat': parts[0],
    'Level': parts[1],
    'Proc': parts[2],
    'ProductType': parts[3],
    'Cycle': parts[4],
    'Track': parts[5],
    'Dir': parts[6],
    'Frame': parts[7],
    'Mode': parts[8],
    'StartDate': parts[11].split('T')[0],   # Extract the date part from the timestamp
    'EndDate': parts[13].split('T')[0],   # Extract the date part from the timestamp
    'Version': parts[10] 
}

print(f"{metadata["Sat"]}_{metadata["Track"]}_{metadata["Dir"]}_{metadata['StartDate']}_{metadata['EndDate']}")

## Add a manually-added zero point if user specifies
## Check if the file f"{filename}_correction.txt" exists, and if so, read the value from it. If not, use a default value of 0.3 meters.
correction_file = directory / f"{filename}_correction.txt"
if os.path.exists(correction_file):
    with open(correction_file, "r") as f:
        zero_point = float(f.read().strip())
    print(f"Using correction point: {zero_point} meters")
    # los_corrected_zeroed = los_corrected - zero_point
    # los_corrected_zeroed = np.where(valid_mask, los_corrected_zeroed, np.nan)
else:
    print(f"No correction point added. \\ Check the file {correction_file}")
    zero_point = None

# Check if the directory has an 'aoi*.geojson' file to crop
aoi_files = list(directory.glob("aoi*.geojson"))
if aoi_files:
    aoi_file = aoi_files[0]
if os.path.exists(correction_file):
    print(f"Using AOI file: {aoi_file}")

NISAR_042_D_20260430_20260512
No correction point added. \ Check the file /Volumes/T9_InSAR/2026-04-14_nevada/NISAR/postseismic/NISAR_L2_PR_GUNW_019_042_D_069_020_4000_SH_20260430T025611_20260430T025646_20260512T025610_20260512T025645_P05023_N_F_J_001_correction.txt


In [32]:
### Helper functions


def print_datasets(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"{name}")
        print(f"    shape = {obj.shape}")
        print(f"    dtype = {obj.dtype}")
        print()


import numpy as np

def make_nisar_valid_mask(mask):
    """
    Return a boolean mask of valid land pixels from a NISAR GUNW mask.

    Parameters
    ----------
    mask : ndarray of uint32
        NISAR GUNW mask from
        science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/mask

    Returns
    -------
    valid : ndarray of bool
        True for valid land pixels, False for water or invalid pixels.

    Notes
    -----
    The lowest byte of the mask is encoded as WRS:

        W = water flag (1 = water, 0 = land)
        R = reference RSLC subswath
        S = secondary RSLC subswath

    Pixels are considered valid if they:
      - are on land
      - have valid subswath numbers (>0) in both images

    Other QA bits (anomalies, ionosphere interpolation, etc.) are
    intentionally ignored.
    """

    mask = np.asarray(mask, dtype=np.uint32)

    # Decode the lowest byte (WRS)
    wrs = mask & 0xFF

    water = wrs // 100
    reference = (wrs % 100) // 10
    secondary = wrs % 10

    return (
        (water == 0) &
        (reference > 0) &
        (secondary > 0)
    )


In [33]:
with h5py.File(gunwfile) as f:
    d = f["science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH"]

    print("Attributes:")
    for k, v in d.attrs.items():
        print(f"{k}: {v}")

Attributes:


In [34]:
import os
import h5py
import numpy as np
import rasterio
from rasterio.transform import from_origin

c = 299792458.0  # Speed of light (m/s)

group = "science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH"
wrapped_group = "science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram"

with h5py.File(gunwfile) as f:

    # Read datasets
    phase = f[f"{group}/unwrappedPhase"][:]
    ionosphere = f[f"{group}/ionospherePhaseScreen"][:]
    nisar_mask = f["science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/mask"][:]

    frequency = f["science/LSAR/GUNW/grids/frequencyA/centerFrequency"][()]
    wavelength = c / frequency

    x = f[f"{group}/xCoordinates"][:]
    y = f[f"{group}/yCoordinates"][:]
    epsg = int(f[f"{group}/projection"][()])

    # The wrapped interferogram is stored as a complex64 array at
    # wrappedInterferogram/HH/wrappedInterferogram, on its OWN grid
    # (finer resolution than the unwrapped group, so it needs its own
    # coordinates/projection/mask). Wrapped phase = angle of the complex
    # value, in radians on (-pi, pi].
    wrapped_complex = f[f"{wrapped_group}/HH/wrappedInterferogram"][:]
    wrapped_x = f[f"{wrapped_group}/HH/xCoordinates"][:]
    wrapped_y = f[f"{wrapped_group}/HH/yCoordinates"][:]
    wrapped_epsg = int(f[f"{wrapped_group}/HH/projection"][()])
    wrapped_nisar_mask = f[f"{wrapped_group}/mask"][:]

# Create land/valid masks
valid_mask = make_nisar_valid_mask(nisar_mask)
wrapped_valid_mask = make_nisar_valid_mask(wrapped_nisar_mask)

# Create an AOI mask if an AOI file is provided.
# The AOI is treated as a rectangle: pixels are kept if their coordinates
# fall within the AOI's bounding box, and discarded otherwise.
if aoi_files:
    import geopandas as gpd

    # Read the AOI GeoJSON file
    aoi_gdf = gpd.read_file(aoi_file)
    minx, miny, maxx, maxy = aoi_gdf.total_bounds

    # Vectorized bounding-box mask for the unwrapped-interferogram grid
    x_in_bounds = (x >= minx) & (x <= maxx)
    y_in_bounds = (y >= miny) & (y <= maxy)
    unwrapped_mask = y_in_bounds[:, None] & x_in_bounds[None, :]

    # # Update the valid mask to include the AOI mask
    # valid_mask &= unwrapped_mask

# Convert phase to LOS displacement
los = phase * wavelength / (4 * np.pi)

# Apply ionosphere correction
phase_corrected = phase - ionosphere
los_corrected = phase_corrected * wavelength / (4 * np.pi)


# Apply water mask
los = np.where(valid_mask, los, np.nan)
los_corrected = np.where(valid_mask, los_corrected, np.nan)


# Wrapped phase (radians) from the complex wrapped interferogram
wrapped_phase = np.angle(wrapped_complex)
wrapped_phase = np.where(wrapped_valid_mask, wrapped_phase, np.nan)

# Build affine transform (unwrapped-interferogram grid)
dx = x[1] - x[0]
dy = y[1] - y[0]

transform = from_origin(
    x.min() - dx / 2,
    y.max() + abs(dy) / 2,
    dx,
    abs(dy),
)

profile = dict(
    driver="GTiff",
    height=los.shape[0],
    width=los.shape[1],
    count=1,
    dtype="float32",
    crs=f"EPSG:{epsg}",
    transform=transform,
    compress="LZW",
    nodata=np.nan,
)
# Write original LOS displacement
with rasterio.open(f"{directory}/{metadata["Sat"]}_{metadata["Track"]}_{metadata["Dir"]}_{metadata['StartDate']}_{metadata['EndDate']}_los.tif", "w", **profile) as dst:
    dst.write(los.astype(np.float32), 1)

# Write ionosphere-corrected LOS displacement
with rasterio.open(f"{directory}/{metadata["Sat"]}_{metadata["Track"]}_{metadata["Dir"]}_{metadata['StartDate']}_{metadata['EndDate']}_los_iono_corrected.tif", "w", **profile) as dst:
    dst.write(los_corrected.astype(np.float32), 1)

# Write ionosphere-corrected LOS displacement with zero point applied (if specified)
if zero_point is not None:
    los_corrected_zeroed = los_corrected - zero_point
    los_corrected_zeroed = np.where(valid_mask, los_corrected_zeroed, np.nan)
    with rasterio.open(f"{directory}/{metadata["Sat"]}_{metadata["Track"]}_{metadata["Dir"]}_{metadata['StartDate']}_{metadata['EndDate']}_los_iono_corrected_zeroed.tif", "w", **profile) as dst:
        dst.write(los_corrected_zeroed.astype(np.float32), 1)

# Build affine transform (wrapped-interferogram grid — different resolution)
wrapped_dx = wrapped_x[1] - wrapped_x[0]
wrapped_dy = wrapped_y[1] - wrapped_y[0]

wrapped_transform = from_origin(
    wrapped_x.min() - wrapped_dx / 2,
    wrapped_y.max() + abs(wrapped_dy) / 2,
    wrapped_dx,
    abs(wrapped_dy),
)

wrapped_profile = dict(
    driver="GTiff",
    height=wrapped_phase.shape[0],
    width=wrapped_phase.shape[1],
    count=1,
    dtype="float32",
    crs=f"EPSG:{wrapped_epsg}",
    transform=wrapped_transform,
    compress="LZW",
    nodata=np.nan,
)

# Write wrapped phase (radians)
with rasterio.open(f"{directory}/{metadata["Sat"]}_{metadata["Track"]}_{metadata["Dir"]}_{metadata['StartDate']}_{metadata['EndDate']}_wrapped_phase.tif", "w", **wrapped_profile) as dst:
    dst.write(wrapped_phase.astype(np.float32), 1)

KeyboardInterrupt: 

## Save as a Geotiff